# Week 6: The Price Is Right

Product price prediction from descriptions. Goal: beat the ~$76 baseline (NLP + LR).

**Order of play:** Data → Baseline → Fine-tune → Evaluate

**Setup:** Add `OPENAI_API_KEY` and `HF_TOKEN` to `.env`

**Run from:** `week6/` directory (or add path so `pricer` imports work)

In [ ]:
import os
import sys
import re
import json
from pathlib import Path

from dotenv import load_dotenv
from huggingface_hub import login
from openai import OpenAI

# Add week6 to path so we can import pricer
week6_root = Path.cwd() if (Path.cwd() / 'pricer').exists() else Path.cwd().parent.parent
sys.path.insert(0, str(week6_root))

from pricer.items import Item
from pricer.evaluator import evaluate

load_dotenv(override=True)
hf_token = os.environ.get("HF_TOKEN", "")
login(hf_token, add_to_git_credential=True)
openai = OpenAI()

## Load data from HuggingFace Hub

In [ ]:
LITE_MODE = True  

username = "ed-donner"
dataset = f"{username}/items_lite" if LITE_MODE else f"{username}/items_full"

train, val, test = Item.from_hub(dataset)

print(f"Loaded {len(train):,} train, {len(val):,} val, {len(test):,} test")

## Baseline predictor (zero-shot)

Optional: test a simple zero-shot LLM predictor before fine-tuning.

In [ ]:
def baseline_pricer(item):
    """Zero-shot price predictor using GPT-4o-mini. Returns price string for evaluate()."""
    prompt = f"Estimate the price of this product in USD. Respond with only the price, no explanation.\n\n{item.summary}"
    r = openai.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": prompt}],
        max_tokens=50,
    )
    return r.choices[0].message.content.strip()

# Uncomment to run baseline (uses API calls)
evaluate(baseline_pricer, test, size=50)

## Prepare JSONL for fine-tuning

In [ ]:
def messages_for(item):
    msg = f"Estimate the price of this product. Respond with the price, no explanation.\n\n{item.summary}"
    return [
        {"role": "user", "content": msg},
        {"role": "assistant", "content": f"${item.price:.2f}"},
    ]

def make_jsonl(items):
    return "\n".join(
        json.dumps({"messages": messages_for(item)}) for item in items
    )

def write_jsonl(items, filename):
    Path(filename).parent.mkdir(parents=True, exist_ok=True)
    Path(filename).write_text(make_jsonl(items), encoding="utf-8")

In [ ]:
# Use 100 train + 50 val (OpenAI recommends 50–100)

fine_tune_train = train[:100]
fine_tune_val = val[:50]

write_jsonl(fine_tune_train, "jsonl/fine_tune_train.jsonl")
write_jsonl(fine_tune_val, "jsonl/fine_tune_validation.jsonl")

print("JSONL files written.")

## Upload and fine-tune (optional)

Uncomment to upload files and start a fine-tuning job. Costs apply.

In [ ]:
with open("jsonl/fine_tune_train.jsonl", "rb") as f:
    train_file = openai.files.create(file=f, purpose="fine-tune")
with open("jsonl/fine_tune_validation.jsonl", "rb") as f:
    val_file = openai.files.create(file=f, purpose="fine-tune")
job = openai.fine_tuning.jobs.create(
    training_file=train_file.id,
    validation_file=val_file.id,
    model="gpt-4.1-nano-2025-04-14",
    suffix="pricer",
    hyperparameters={       
        "n_epochs": 1,
        "batch_size": 1,
    }
)

In [ ]:
job_id = openai.fine_tuning.jobs.list(limit=1).data[0].id


In [ ]:
job_id


In [ ]:
openai.fine_tuning.jobs.retrieve(job_id)

In [ ]:
openai.fine_tuning.jobs.list_events(fine_tuning_job_id=job_id, limit=10).data

## Fine-tuned predictor

Replace `FINE_TUNED_MODEL` with your model ID after the job completes.

In [ ]:
FINE_TUNED_MODEL = "ft:gpt-4.1-nano-2025-04-14:kimanij:pricer:DHUVh2mW"  # Replace with your model ID

def fine_tuned_pricer(item):
    prompt = f"Estimate the price of this product. Respond with the price, no explanation.\n\n{item.summary}"
    r = openai.chat.completions.create(
        model=FINE_TUNED_MODEL,
        messages=[{"role": "user", "content": prompt}],
        max_tokens=50,
    )
    return r.choices[0].message.content.strip()

evaluate(fine_tuned_pricer, test, size=50)
